# 10. Conclusiones, limitaciones y checklist

Este capítulo **lee las tablas generadas en los capítulos anteriores** y resume los hallazgos; las cifras salen de esas tablas, no están escritas a mano.

In [ ]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

import platform
res = pd.read_csv(TABLES / "resultados_holdout.csv", index_col=0)
ic = pd.read_csv(TABLES / "intervalos_bootstrap_logistica.csv", index_col=0)
dif = pd.read_csv(TABLES / "diferencias_vs_dummy.csv", index_col=0)
audit = pd.read_csv(TABLES / "auditoria_nota_critica.csv")
pop = pd.read_csv(TABLES / "decision_poblacion.csv").iloc[0]
tra = pd.read_csv(TABLES / "decision_transformacion.csv").iloc[0]
rank = pd.read_csv(TABLES / "ranking_univariado.csv")
falt = pd.read_csv(TABLES / "faltantes_por_columna.csv", index_col=0)
obj = pd.read_csv(TABLES / "objetivo_distribucion.csv")
orr = pd.read_csv(TABLES / "odds_ratios.csv")
sub = pd.read_csv(TABLES / "desempeno_subgrupos.csv")
umb = pd.read_csv(TABLES / "umbral_alternativo.csv", index_col=0)
part = pd.read_csv(TABLES / "particion_encuentros.csv")
lr, dm = res.loc["Regresión logística"], res.loc["Dummy (clase mayoritaria)"]

## Resumen ejecutivo

In [ ]:
n_train, n_test = (part["conjunto"] == "train").sum(), (part["conjunto"] == "test").sum()
md_out(
    f"Se construyó una línea base para predecir si un encuentro de un paciente con diabetes terminará en **reingreso antes de 30 días**, "
    f"con {es(n_train + n_test, 0)} encuentros y una partición agrupada por paciente ({es(n_train, 0)} de entrenamiento; {es(n_test, 0)} de prueba). "
    f"En el holdout, la regresión logística obtiene ROC-AUC {es(lr['ROC-AUC'], 3)} "
    f"(IC 95 %: {es(ic.loc['ROC-AUC', 'IC 95 % inferior'], 3)}–{es(ic.loc['ROC-AUC', 'IC 95 % superior'], 3)}) y Average Precision {es(lr['Average Precision'], 3)}. "
    f"Con el umbral 0,5, su accuracy es {es(lr['Accuracy'], 3)}, igual a la línea trivial ({es(dm['Accuracy'], 3)}), "
    f"pero el recall es apenas {es(lr['Recall'], 3)}. Por tanto, el modelo aporta una señal de ordenamiento modesta, pero el umbral estándar **no es útil para detectar la clase positiva**; "
    "cualquier cambio de umbral debe decidirse posteriormente con costos y criterios clínicos definidos, no observando el test."
)

## Respuestas a las preguntas del enunciado

In [ ]:
top_var = rank.head(3)
faltan = falt["faltantes_%"].sort_values(ascending=False)
preg = [
 ("¿Qué se predice y para qué?", "Reingreso hospitalario en menos de 30 días; busca apoyar la priorización académica de seguimiento sin convertirse en una decisión clínica automática."),
 ("¿Cuántos datos hay y cuántos son independientes?", f"{es(n_train + n_test, 0)} encuentros; los pacientes son las entidades independientes y se usaron para partir y validar."),
 ("¿Qué tan balanceadas están las clases?", f"Reingreso <30: {es(obj.loc[1, '%'], 1)} %; clase 0 (>30 o NO): {es(obj.loc[0, '%'], 1)} %."),
 ("¿Qué problemas de calidad hay?", "Faltantes altos en " + ", ".join(f"`{v}` ({es(faltan[v], 0)} %)" for v in faltan.index[:3]) + "; altas terminales; variables constantes."),
 ("¿Qué variables se asocian más al reingreso?", ", ".join(f"`{r['variable']}` (AUC univariado {es(r['AUC univariado'], 3)})" for _, r in top_var.iterrows()) + "."),
 ("¿Hay fuga de datos?", f"No se detectó ninguna variable que por sí sola reproduzca el objetivo (máximo AUC univariado {es(rank['AUC univariado'].max(), 3)}). "
                         + ("Se excluyeron las altas por fallecimiento u hospicio, que no pueden reingresar." if bool(pop['excluir_terminales']) else "")),
 ("¿Qué preprocesamiento se aplicó?", f"Pipeline con imputación, `Missing` para categóricas, categorías raras agrupadas, estandarización y {'`log1p`' if bool(tra['usar_log1p']) else 'sin `log1p`'} en conteos."),
 ("¿Cuánto mejora el modelo sobre la línea base?", f"ROC-AUC {es(lr['ROC-AUC'], 3)} frente a 0,5 y Average Precision {es(lr['Average Precision'], 3)} frente a {es(dm['Average Precision'], 3)}; el accuracy no cambia y el recall con umbral 0,5 es {es(lr['Recall'], 3)}."),
 ("¿Se activa la nota crítica (accuracy ≥ 80–90 %)?", "No." if lr["Accuracy"] < 0.80 else "Sí; ver la auditoría del capítulo 9."),
]
# Se muestra el texto completo de cada respuesta (sin el recorte de 70 caracteres del resto del libro).
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(preg, columns=["pregunta", "respuesta"]))

## Limitaciones

In [ ]:
rango_sub = sub["AUC"].max() - sub["AUC"].min()
md_out("- **Observacional y de un solo sistema:** datos de hospitales de EE. UU. entre 1999 y 2008; las asociaciones no son causales y no se extrapolan sin más a otros países ni épocas.\n"
       "- **Sin fecha por encuentro:** no se pudo estudiar deriva temporal; la partición no es cronológica.\n"
       "- **Objetivo y seguimiento incompletos:** un reingreso en otro hospital no aparece en los datos.\n"
       f"- **Poder predictivo {'modesto' if lr['ROC-AUC'] < 0.75 else 'razonable'}:** con un AUC de {es(lr['ROC-AUC'], 3)}, el modelo sirve para **priorizar**, no para decidir un caso individual.\n"
       f"- **Diferencias entre subgrupos:** el AUC varía {es(rango_sub, 3)} entre subgrupos de sexo, raza y edad; hay que auditarlo antes de cualquier uso.\n"
       "- **Faltantes:** el mecanismo MNAR no puede descartarse en variables como el peso o la especialidad médica.\n"
       "- **Línea base lineal:** una regresión logística no captura interacciones ni no linealidades; es el punto de partida, no el techo.")

## Usos apropiados y consideraciones éticas

Los registros están anonimizados, pero contienen atributos sensibles como raza,
sexo y edad. El conjunto corresponde a una población hospitalaria específica de
Estados Unidos entre 1999 y 2008 y puede reflejar desigualdades de acceso,
prácticas históricas y sesgos de documentación. Un modelo con rendimiento
distinto entre subgrupos podría distribuir de forma desigual los recursos de
seguimiento.

Este trabajo es académico: no está validado externamente, no debe usarse para
negar atención y no constituye una herramienta clínica. Antes de cualquier uso
real serían necesarias revisión clínica, validación contemporánea y externa,
evaluación de sesgo, seguridad, privacidad y monitoreo de calibración.

## Próximos pasos

En una fase posterior se podrían comparar modelos no lineales, revisar umbrales
según costos acordados con expertos y validar en datos externos. Esas tareas no
forman parte de este entregable base.

## Checklist del enunciado

In [ ]:
checklist = [
 ("Fuente, DOI, licencia, población, periodo y unidad de observación", "Cap. 1"),
 ("Target <30 consistente", "Caps. 1, 3, 9 y 10"),
 ("n/p, clases, pacientes y encuentros por paciente", "Cap. 2"),
 ("Calidad: faltantes, mecanismo prudente, duplicados, outliers e inconsistencias", "Cap. 2"),
 ("Split 80/20 estratificado y agrupado; cero pacientes compartidos", "Caps. 1 y 7"),
 ("EDA objetivo, univariado y bivariado con tamaños de efecto/FDR", "Caps. 3–5"),
 ("Pearson, Spearman, hexbin, χ², V de Cramér e información mutua", "Caps. 5–6"),
 ("VIF, PCA, varianza explicada e Isolation Forest", "Cap. 6"),
 ("Auditoría variable por variable y AUC univariado", "Cap. 7"),
 ("Pipeline/ColumnTransformer sin fuga", "Cap. 8"),
 ("DummyClassifier y regresión logística", "Cap. 9"),
 ("Accuracy, precision, recall, F1, ROC-AUC, AP, matrices, ROC y PR", "Cap. 9"),
 ("Calibración y Brier score", "Cap. 9"),
 ("Bootstrap por paciente e IC por métrica", "Cap. 9"),
 ("Curva de aprendizaje agrupada e interpretación de odds ratios", "Cap. 9"),
 ("Referencias reales, requirements y Jupyter Book reproducible", "Archivos raíz"),
]
checklist_table = pd.DataFrame([("✓", a, b) for a, b in checklist], columns=["", "requisito", "evidencia"])
checklist_table.to_csv(TABLES / "checklist_entregable.csv", index=False)
with pd.option_context("display.max_colwidth", None):
    display(checklist_table)

## Reproducibilidad: semillas y versiones

In [ ]:
import sklearn, scipy, matplotlib
versiones = pd.DataFrame({"paquete": ["Python", "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "seaborn"],
                          "versión": [platform.python_version(), np.__version__, pd.__version__, scipy.__version__, sklearn.__version__, matplotlib.__version__, sns.__version__]})
display(versiones)
md_out(f"**Semilla global:** `RANDOM_STATE = {RANDOM_STATE}` (partición, pliegues de validación, bootstrap y muestreos de gráficos). "
       f"**Partición:** `tables/particion_encuentros.csv`. **Validación:** `StratifiedGroupKFold` con {N_SPLITS} pliegues agrupados por `{GROUP_COL}`. "
       "Las tablas de resultados están en la carpeta `tables/` y las figuras en `figures/`.")

## Referencias

```{bibliography}
:style: plain
```